# Исследование интернет-магазине «Стримчик»

<b>Цель:</b> проанализировать данные и проверьте некоторые гипотезы, которые могут помочь бизнесу вырасти.

<b>План работы:</b>

1) Обзор данных

2) Предобработка данных

3) Исследовательский анализ данных

4) Составление портрета пользователя каждого региона

5) Проверка гипотез

6) Общий вывод

## Обзор данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import seaborn as sns

In [ ]:
data = pd.read_csv('/datasets/games.csv')
data.head()

In [ ]:
data.info()

In [ ]:
data.hist(figsize=(5, 7))
plt.show()

<div class="alert alert-info">
    <b>Вывод:</b> 
    
    в данных есть большое колличество пропусков, данные с продажами имеют большие выбросы, оценка пользователя имеет строчный тип данных, год выпуска имеею тип с влающей точкой.
</div>

## Предобработка данных

In [ ]:
data.columns = data.columns.str.lower()
data.columns

In [ ]:
data.isna().sum()

In [ ]:
data = data.dropna(subset=['name', 'year_of_release', 'genre'])

tbd_indices = data['user_score'] == 'tbd'
data['user_score'] = pd.to_numeric(data['user_score'], errors='coerce')
data.loc[~tbd_indices, 'user_score'] = data.loc[~tbd_indices, 'user_score'].fillna(data['user_score'].median())

data['critic_score'] = data.groupby('genre')['critic_score'].transform(lambda x: x.fillna(x.median()))
data['rating'] = data.groupby('genre')['rating'].transform(lambda x: x.fillna(x.mode()[0]))

data.isna().sum()

In [ ]:
data['year_of_release'] = data['year_of_release'].astype(int)
data['year_of_release'].dtype

In [ ]:
data['year_of_release'].unique()

In [ ]:
data.duplicated().sum()

In [ ]:
data['total_sales'] = (data['na_sales'] + data['eu_sales'] + data['jp_sales'] + data['other_sales']).round().astype(int)
data.head()

In [1]:
data[data.duplicated(subset=['name', 'platform'], keep=False)]

<div class="alert alert-info">
    <b>Вывод:</b> 
    
    1) Дубликаты отсутствуют
    
    2) Пропуски в оценке критиков заполним медианой по жанру, это оптимальное заполнение, потому что одни жанры пользуются большей популярность и лучше взять медиану по жанру.
    
    3) Пропуски в оценке организации заполним модой по жанру, это оптимальное заполнение, потому что одни жанры получаю более серьезное возрастные ограничения, например хорроры
    
    4) Пропуски в оценке пользователей заполним медианой по жанру, а значения tbd означает, что еще данные 
    отсутствуют, заполним Nan, это лучший вариант, потому что игроки могут по разному оценить игру, так в будущем, когда появится рейтинг его можно легко будет заполнить
    
    5) Изменили тип столбца оценка пользователя, на число с плавающей точкой, так как там присутствую только числа
    
    6) Изменили тип столбца с годовым выпуска на целочисленный
    
    7) Причину появления пропусков могут быть: 	
    
    1.	Ошибки при сборе данных
    
	2.	Неполные данные
    
	3.	Человеческий фактор
    
	4.	Различия в форматах и стандартах
</div>

## Исследовательский анализ

In [ ]:
plt.figure(figsize=(12, 6))
data['year_of_release'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Количество игр по годам выпуска')
plt.xlabel('Год выпуска')
plt.ylabel('Количество игр')
plt.show()

<div class="alert alert-info">
1)Рост колличества игр начался с 2000-х
    
2)Спад в 2012 может быть связан: 1. с переходом на новые покаления платформ 2. с перенасыщение рынка в предыдущие годы
</div>

In [ ]:
platform_sales = data.groupby('platform')['total_sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
platform_sales.plot(kind='bar', color='orange')
plt.title('Суммарные продажи по платформам')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.show()

<div class="alert alert-info">
1)Лидер по продажам - PS2
    
2)X360, PS3, Wii, PS, DS - являются лидерами по продажам
</div>

In [ ]:
top_platforms = platform_sales.head(5).index
data_top_platforms = data[data['platform'].isin(top_platforms)]
plt.figure(figsize=(12, 6))
sns.histplot(data=data_top_platforms, x='year_of_release', hue='platform', multiple='stack', bins=20)
plt.title('Распределение продаж по платформам и годам')
plt.xlabel('Год выпуска')
plt.ylabel('Количество игр')
plt.show()

<div class="alert alert-info">
1)Рост в 2005-2006 годах обусловлен появлением новых платформ
    
2)Спад в 2012 году обусловен вытеснением конкурентов платформы PS3
    
3)На данным момент лидер с более чем половиной рынка является - PS3, который держится стабильно, а такой конкурент, как X360 теряет продажи
</div>

In [ ]:
platform_lifecycle = data_top_platforms.groupby(['platform', 'year_of_release'])['total_sales'].sum().reset_index()
plt.figure(figsize=(12, 6))
for platform in top_platforms:
    platform_data = platform_lifecycle[platform_lifecycle['platform'] == platform]
    plt.plot(platform_data['year_of_release'], platform_data['total_sales'], label=platform)
plt.title('Жизненный цикл платформ')
plt.xlabel('Год выпуска')
plt.ylabel('Продажи (млн копий)')
plt.legend()
plt.show()

<div class="alert alert-info">
1)Поколения платформ меняются около 7 лет
    
2)Платформы остаются на пике своей актуальности около 5 лет
</div>

In [ ]:
actual_period = data[data['year_of_release'] >= 2012]

actual_platform_sales = actual_period.groupby('platform')['total_sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
actual_platform_sales.plot(kind='bar', color='green')
plt.title('Продажи платформ за актуальный период (2012-2016)')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.show()

<div class="alert alert-info">
1)За последние 5 лет PS4 лидирует по продажам и является наиболее потенциальная
    
2) Так же стоит отменить платформы 3DS и XOne как наиболеепотенциальные
    
2)PS3 не сильно отстает от PS4 по колличеству продаж
    
3)X360 находится в тройке лидеров по продажам
</div>

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=actual_period, x='platform', y='total_sales', order=actual_platform_sales.index)
plt.title('Распределение глобальных продаж по платформам')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=actual_period, x='platform', y='total_sales', order=actual_platform_sales.index)
plt.title('Распределение глобальных продаж по платформам')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.ylim(0,3)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=actual_period, x='platform', y='total_sales', order=actual_platform_sales.index)
plt.ylim(0, actual_period['total_sales'].quantile(0.99))
plt.title('Распределение глобальных продаж по платформам')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=actual_period, x='platform', y='total_sales', order=actual_platform_sales.index)
plt.ylim(0, actual_period['total_sales'].quantile(0.75))
plt.title('Распределение глобальных продаж по платформам')
plt.xlabel('Платформа')
plt.ylabel('Продажи (млн копий)')
plt.show()

<div class="alert alert-info">
1)PS3 имеет самую продаваемую игру
    
2)3DS имеет несколько игр которые хорошо продаются, но в основном игры на этой платформе не имеют продаж
    
3)Игры имеющие более 10 млн продаж являются редкими и успешными 
    
4) Нормальный диапозон продаж 2 млн экземпляров
</div>

In [ ]:
top_platforms = actual_period['platform'].value_counts().head().index

# Для каждой платформы строим графики и рассчитываем корреляцию Пирсона
for platform in top_platforms:
    platform_data = actual_period[actual_period['platform'] == platform]
    
    # График для оценки критиков и продаж
    plt.figure(figsize=(12, 6))
    sns.scatterplot(data=platform_data, x='critic_score', y='total_sales', alpha=0.6)
    plt.title(f'Связь между отзывами критиков и продажами ({platform})')
    plt.xlabel('Оценка критиков')
    plt.ylabel('Продажи (млн копий)')
    plt.show()
    
    # График для оценки пользователей и продаж
    plt.figure(figsize=(12, 6))
    sns.scatterplot(data=platform_data, x='user_score', y='total_sales', alpha=0.6)
    plt.title(f'Связь между отзывами пользователей и продажами ({platform})')
    plt.xlabel('Оценка пользователей')
    plt.ylabel('Продажи (млн копий)')
    plt.show()
    
    # Корреляция для платформы
    critic_corr = platform_data['critic_score'].corr(platform_data['total_sales'])
    user_corr = platform_data['user_score'].corr(platform_data['total_sales'])

    print(f'Платформа {platform}:')
    print(f'Корреляция между оценками критиков и продажами: {critic_corr:.2f}')
    print(f'Корреляция между оценками пользователей и продажами: {user_corr:.2f}')
    print('-' * 50)

<div class="alert alert-info">
1)Продажи игры коррелируют с оценкой критиков
    
2)Продажи игры не коррелируют с оценкой пользователей
</div>

In [ ]:
genre_sales = actual_period.groupby('genre')['total_sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
genre_sales.plot(kind='bar', color='purple')
plt.title('Продажи по жанрам')
plt.xlabel('Жанр')
plt.ylabel('Продажи (млн копий)')
plt.show()

# Распределение игр по жанрам
plt.figure(figsize=(12, 6))
sns.boxplot(data=actual_period, x='genre', y='total_sales', order=genre_sales.index)
plt.title('Распределение продаж по жанрам')
plt.xlabel('Жанр')
plt.ylabel('Продажи (млн копий)')
plt.show()

<div class="alert alert-info">
1)Sports, Shooter, Platform относительно других жанров имеют базовые продажи
    
2) Самая популярная игр имеет жанр - Sports
    
3) Топ-3 самых продаваемый жанров игр - Action, Sports, Shooter
</div>

<div class="alert alert-info">
    <b>Вывод:</b> 
    
    1) Поколения платформ меняются около 7 лет
    
    2) Платформы остаются на пике своей актуальности около 5 лет
    
    3) Платформы с лучшим потенциалом - PS4, XOne.
    
    4) Игры имеющие более 10 млн продаж являются редкими и успешными
    
    5) Продажи игры коррелируют с оценкой критиков и не коррелируют с оценкой пользователей
    
    6) Топ-3 самых продаваемый жанров игр - Action, Sports, Shooter
    
    7) Sports, Shooter, Platform относительно других жанров имеют базовые продажи
</div>

## Составление портрета пользователя каждого региона

In [ ]:
data.info()

In [ ]:
# Топ-5 платформ по регионам (актуальный период)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, region, title in zip(axes, ['na_sales', 'eu_sales', 'jp_sales'], ['Северная Америка', 'Европа', 'Япония']):
    top_platforms = actual_period.groupby('platform')[region].sum().sort_values(ascending=False).head(5)
    top_platforms.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90, colors=plt.cm.Set3.colors)
    ax.set_title(f'Топ-5 платформ в {title}')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

# Топ-5 жанров по регионам (актуальный период)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, region, title in zip(axes, ['na_sales', 'eu_sales', 'jp_sales'], ['Северная Америка', 'Европа', 'Япония']):
    top_genres = actual_period.groupby('genre')[region].sum().sort_values(ascending=False).head(5)
    top_genres.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90, colors=plt.cm.Pastel1.colors)
    ax.set_title(f'Топ-5 жанров в {title}')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

# Влияние рейтинга ESRB (актуальный период)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, region, title in zip(axes, ['na_sales', 'eu_sales', 'jp_sales'], ['Северная Америка', 'Европа', 'Япония']):
    esrb_impact = actual_period.groupby('rating')[region].sum().sort_values(ascending=False)
    esrb_impact.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90, colors=plt.cm.Set2.colors)
    ax.set_title(f'Влияние рейтинга ESRB в {title}')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

<div class="alert alert-info">
    <b>Вывод:</b> 
    
    1) В Северной Америкеи 1. платформам X360, PS2, Wii 2. жанрам Action, Sports, Shooter 3. рейтингу ESRB E и T- принадлежат 2/3 рынка
    
    2) в Европе 1. платформам PS3, PS2, X360 2. жанрам Action, Sports, Shooter 3. рейтингу ESRB E и T- принадлежат 2/3 рынка
    
    3) В Японии 1. платформам DS, PS2, PS. жанру Role-Playing и Action 3. рейтингу ESRB E и T - принадлежат 2/3 рынка
</div>

## Проверка гипотез

Средние рейтинги платформ Xbox One и PC одинаковые:

	Гипотеза H0: Средние рейтинги платформ Xbox One и PC одинаковые
    
	Гипотеза H1: Средние рейтинги платформ Xbox One и PC разные

In [ ]:
alpha = 0.05

xbox_user_scores = actual_period[actual_period['platform'] == 'XOne']['user_score'].dropna()
pc_user_scores = actual_period[actual_period['platform'] == 'PC']['user_score'].dropna()

t_stat, p_value = st.ttest_ind(xbox_user_scores, pc_user_scores, equal_var=False)  # Учтем возможное различие дисперсий
print(p_value)
if p_value < alpha:
    print("Отвергаем нулевую гипотезу: Средние пользовательские рейтинги платформ XOne и PC разные.")
else:
    print("Не отвергаем нулевую гипотезу: Средние пользовательские рейтинги платформ XOne и PC одинаковые.")

Средние рейтинги жанров Action и Sports разные:

	Гипотеза H1: Средние рейтинги жанров Action и Sports одинаковые
    
	Гипотеза H1: Средние рейтинги жанров Action и Sports разные

In [ ]:
alpha = 0.05

action_scores = actual_period[actual_period['genre'] == 'Action']['user_score'].dropna()
sports_scores = actual_period[actual_period['genre'] == 'Sports']['user_score'].dropna()

t_stat, p_value = st.ttest_ind(action_scores, sports_scores, equal_var=False)
print(p_value)
if p_value < alpha:
    print("Отвергаем нулевую гипотезу: Средние пользовательские рейтинги жанров Action и Sports разные.")
else:
    print("Не отвергаем нулевую гипотезу: Средние пользовательские рейтинги жанров Action и Sports одинаковые.")

<div class="alert alert-info">
1) Средние пользовательские рейтинги платформ Xbox One и PC одинаковые
    
2) Средние пользовательские рейтинги жанров Action и Sports разные.
</div>

## Общий вывод

<div class="alert alert-info">
    <b>Вывод:</b> 
    
    1) В Северной Америкеи 1. платформам X360, PS2, Wii 2. жанрам Action, Sports, Shooter 3. рейтингу ESRB E и T- принадлежат 2/3 рынка
    
    2) в Европе 1. платформам PS3, PS2, X360 2. жанрам Action, Sports, Shooter 3. рейтингу ESRB E и T- принадлежат 2/3 рынка
    
    3) В Японии 1. платформам DS, PS2, PS. жанру Role-Playing и Action 3. рейтингу ESRB E и T - принадлежат 2/3 рынка
    
    4) Средние пользовательские рейтинги платформ Xbox One и PC одинаковые
    
    5) Средние пользовательские рейтинги жанров Action и Sports разные.
    
    6) Платформы остаются на пике своей актуальности около 5 лет
    
    7) Платформы с лучшим потенциалом - PS4, XOne.
    
    8) Продажи игры коррелируют с оценкой критиков и не коррелируют с оценкой пользователей
    
    9) Топ-3 самых продаваемый жанров игр - Action, Sports, Shooter
    
    10) Sports, Shooter, Platform относительно других жанров имеют базовые продажи
</div>